In [1]:
# CELL 1: Environment setup and load Stage 3 triples
import json
import pandas as pd
import networkx as nx
from pathlib import Path
from collections import Counter

RESULTS_DIR = Path("../data/results")
TRIPLES_FILE = RESULTS_DIR / "community_triples.json"
GRAPH_FILE = RESULTS_DIR / "knowledge_graph.graphml"
NODE_FILE = RESULTS_DIR / "knowledge_graph_nodes.csv"
EDGE_FILE = RESULTS_DIR / "knowledge_graph_edges.csv"
METRICS_FILE = RESULTS_DIR / "knowledge_graph_metrics.json"

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print(f"Loaded triples from {len(community_triples)} communities")

Loaded triples from 18 communities


In [2]:
# CELL 2: Node type inference based on semantic content
def infer_node_type(node: str) -> str:
    node = str(node).lower().strip()
    if "network_flow" in node:
        return "entity"
    if "service" in node:
        return "service"
    if "activity" in node:
        return "attack_activity"
    if any(x in node for x in ["behavior", "connection", "pattern", "traffic"]):
        return "network_behavior"
    if "duration" in node:
        return "temporal_feature"
    return "entity"

In [3]:
# CELL 3: Construct directed knowledge graph from semantic triples
G = nx.DiGraph()
edge_weights = {}
total_triples = 0
invalid_triples = 0

for cid, triples in community_triples.items():
    for t in triples:
        s = str(t.get("subject", "")).strip().lower()
        r = str(t.get("relation", "")).strip().lower()
        o = str(t.get("target", t.get("object", ""))).strip().lower()
        
        if not s or not r or not o:
            invalid_triples += 1
            continue
            
        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1

for (src, rel, tgt), weight in edge_weights.items():
    G.add_node(src, type=infer_node_type(src))
    G.add_node(tgt, type=infer_node_type(tgt))
    G.add_edge(src, tgt, relation=rel, weight=weight)

print(f"Graph constructed: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Processed {total_triples} triples | Skipped {invalid_triples} invalid")

Graph constructed: 12 nodes, 11 edges
Processed 72 triples | Skipped 0 invalid


In [4]:
# CELL 4: Export graph structure to standard formats
nx.write_graphml(G, GRAPH_FILE)

nodes = [{"node": n, "type": G.nodes[n].get("type", "unknown"), "degree": d} for n, d in G.degree()]
pd.DataFrame(nodes).sort_values("degree", ascending=False).to_csv(NODE_FILE, index=False)

edges = [{"source": u, "target": v, "relation": data["relation"], "weight": data["weight"]} 
         for u, v, data in G.edges(data=True)]
pd.DataFrame(edges).sort_values("weight", ascending=False).to_csv(EDGE_FILE, index=False)

print("Exported graph to GraphML, nodes.csv, edges.csv")

Exported graph to GraphML, nodes.csv, edges.csv


In [5]:
# CELL 5: Compute and save reproducible graph metrics
relation_dist = Counter([data["relation"] for _, _, data in G.edges(data=True)])
type_dist = Counter([G.nodes[n].get("type", "unknown") for n in G.nodes()])

metrics = {
    "construction_method": "LLM-extracted (subject, relation, target) triples -> NetworkX DiGraph",
    "total_triples_processed": total_triples,
    "invalid_triples_skipped": invalid_triples,
    "unique_nodes": G.number_of_nodes(),
    "unique_edges": G.number_of_edges(),
    "average_degree": float(sum(dict(G.degree()).values()) / max(G.number_of_nodes(), 1)),
    "relation_distribution": dict(relation_dist),
    "node_type_distribution": dict(type_dist)
}

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved. Stage 4 complete.")
print("Relation distribution:", dict(relation_dist))
print("Node type distribution:", dict(type_dist))

Metrics saved. Stage 4 complete.
Relation distribution: {'indicates_activity': 2, 'targets_service': 5, 'shows_behavior': 2, 'has_duration': 2}
Node type distribution: {'entity': 2, 'attack_activity': 2, 'service': 5, 'network_behavior': 2, 'temporal_feature': 1}
